In [1]:
import pandas as pd
movies_df = pd.read_csv("../../data/raw/rotten_tomatoes/rotten_tomatoes_raw.csv")
movies_df.head()


,Title,Summary,Director,Producer,Distributor,Genre,Original Language,Release Date(Streaming),Runtime,Critics score,...,Rating,Release Date (Theaters),Rerelease Date (Theaters),Release Date (Streaming),Box Office (Gross USA),Sound Mix,Aspect Ratio,Release Date (TV),Release Date (DVD),Creator
0,Avengers: Endgame,"Adrift in space with no food or water, Tony St...","Anthony Russo, Joe Russo",Kevin Feige,Walt Disney Pictures,"Action, Adventure, Fantasy, Sci-Fi",English,NaN,3h 1m,94%,...,PG-13 (Sequences of Sci-Fi Violence|Action|Som...,"Apr 26, 2019, Wide","Sep 25, 2026","Jul 30, 2019",$858.4M,"Dolby Digital, SDDS, Dolby Atmos",Digital 2.39:1,NaN,NaN,NaN
1,Parasite,A scientist doing research on the paranormal d...,NaN,NaN,NaN,"Horror, Sci-Fi",English,NaN,1h 30m,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Once Upon a Time... In Hollywood,Actor Rick Dalton gained fame and fortune by s...,Quentin Tarantino,"David Heyman, Shannon McIntosh, Quentin Tarantino","Columbia Pictures, Sony Pictures Entertainment","Comedy, Drama",English,NaN,2h 41m,86%,...,R (Language Throughout|Drug Use|Sexual Referen...,"Jul 26, 2019, Wide",NaN,"Aug 27, 2019",$142.4M,"Dolby Atmos, Dolby Digital, SDDS, Datasat",Digital 2.39:1,NaN,NaN,NaN
3,Joker,A drama directed by Yuri Kuzmenko featuring Va...,Yuri Kuzmenko,NaN,NaN,"Action, Adventure",Russian,NaN,1h 30m,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# data before cleaned 
print(len(movies_df))

79130


In [3]:
# drop lines that have all nulls
movies_df = movies_df.dropna(how="all")

In [4]:
print(len(movies_df))

37076


In [5]:
movies_df.isna().sum().sort_values(ascending=False)

Release Date(Streaming)      37076
Creator                      37007
Release Date (DVD)           36565
Release Date (TV)            36042
Aspect Ratio                 33925
Sound Mix                    33602
Box Office (Gross USA)       32784
Release Date (Streaming)     29044
Release Date (Theaters)      28797
Distributor                  27878
Rerelease Date (Theaters)    27447
Critics score                26813
Rating                       22593
Audience score               22392
Production Co                17371
Producer                     11544
Screenwriter                 10607
Summary                       3656
Runtime                       1563
Original Language             1548
Director                      1471
Genre                          851
Audience reviews                 3
Critics reviews                  1
Title                            0
dtype: int64

In [6]:
len(movies_df)

37076

### because we have so many null values we have decided to keep only the important columns and also remove lines where these important cols are null

In [7]:
movies_df = movies_df[
    movies_df[
        ['Release Date (Theaters)', 'Critics score', 'Audience score']
    ].notnull().all(axis=1)
]

### We will keep only these cols because the others have many null values and the informations they give are not so important for example box office already existed in numbers so no need to have it also in rotten 

In [8]:
important_cols = [
    'Title',
    'Audience score',
    'Critics reviews',
    'Critics score',
    'Audience reviews',
    'Release Date (Theaters)',
    'Genre',
    'Director',
    'Original Language',
    'Runtime',
    'Summary'
]

movies_df = movies_df[important_cols]

In [9]:
movies_df.isna().sum()

Title                       0
Audience score              0
Critics reviews             0
Critics score               0
Audience reviews            0
Release Date (Theaters)     0
Genre                       1
Director                    8
Original Language           9
Runtime                    19
Summary                    21
dtype: int64

### remove rows that have null values 


In [10]:
movies_df = movies_df[
    movies_df[
        ['Genre', 'Director', 'Original Language','Runtime','Summary']
    ].notnull().all(axis=1)
]


In [11]:
movies_df.isna().sum()

Title                      0
Audience score             0
Critics reviews            0
Critics score              0
Audience reviews           0
Release Date (Theaters)    0
Genre                      0
Director                   0
Original Language          0
Runtime                    0
Summary                    0
dtype: int64

In [12]:
movies_df.columns = movies_df.columns.str.lower()


In [13]:
# show only number of reviews instead of number of reviews + "Reviews" and change it to float Ex : "30 Reviews" ===> 30
movies_df["critic_reviews_count"] = (
    movies_df["critics reviews"]
    .str.replace(" Reviews", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

In [14]:
# Remove percentage and change it to float  Ex ! "30%" ===> 30

movies_df["critic_score_percentage"] = (
    movies_df["critics score"]
    .str.replace("%","",regex=False)
    .astype(float)
)
movies_df["audience_score_percentage"] = (
    movies_df["audience score"]
    .str.replace("%","",regex=False)
    .astype(float)
)


In [15]:
# split audience reviews into thre columns 
# Example 1: 
# before: audience reviews[4] = 250+ Verified Ratings
# After : is_audience_rating_verified[4] = True (cause Verified exist in review)
#         audience_rating_count[4] = 250 
#        
# Example 2: 
# before: audience reviews[3] = Fewer than 50 Ratings
# After : is_audience_rating_verified[3] = False (cause Verified don't exist in review)
#         audience_rating_count[3] = 50 
#         
movies_df["is_audience_rating_verified"] = movies_df["audience reviews"].str.contains(
    "Verified",
    case=False,
    na=False
)
movies_df["audience_rating_count"] = (
    movies_df["audience reviews"]
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")[0]
    .astype("Int64")
)



movies_df[["audience reviews","audience_rating_count","is_audience_rating_verified"]]

,audience reviews,audience_rating_count,is_audience_rating_verified
0,"1,000+ Verified Ratings",1000,True
2,"25,000+ Verified Ratings",25000,True
5,"50,000+ Verified Ratings",50000,True
6,"25,000+ Verified Ratings",25000,True
7,"25,000+ Verified Ratings",25000,True
...,...,...,...
79065,"25,000+ Ratings",25000,False
79090,"1,000+ Ratings",1000,False
79091,"10,000+ Ratings",10000,False
79116,500+ Ratings,500,False


In [16]:
# I have found that their is some values that are not in correct format so I deleted rows that don't have the correct one
movies_df = movies_df[
    movies_df["release date (theaters)"].astype("string").str.match(
        r"^[A-Z][a-z]{2} \d{1,2}, \d{4}(, (Limited|Wide|Original))?$",
        na=False
    )
]

In [17]:


# Remove the release type only when it exists
movies_df["release date (theaters)"] = movies_df[
    "release date (theaters)"
].str.replace(
    r",\s*(?:Wide|Limited|Original)\s*$",
    "",
    regex=True
)

# Convert to datetime; invalid dates become NaT
movies_df["release date (theaters)"] = pd.to_datetime(
    movies_df["release date (theaters)"],
    format="%b %d, %Y",
    errors="coerce"
)

# Extract year, month, and day
movies_df["release_date_theaters_year"] = (
    movies_df["release date (theaters)"].dt.year
)

movies_df["release_date_theaters_month"] = (
    movies_df["release date (theaters)"].dt.month
)

movies_df["release_date_theaters_day"] = (
    movies_df["release date (theaters)"].dt.day
)

In [18]:
movies_df["runtime"].isna().sum()

np.int64(0)

In [19]:

movies_df["runtime_hour"] = (
    pd.to_numeric(
        movies_df["runtime"].str.extract(r"(\d+)\s*h", expand=False),
        errors="coerce"
    )
    .fillna(0)
    .astype("Int64")
)

movies_df["runtime_minutes"] = (
    pd.to_numeric(
        movies_df["runtime"].str.extract(r"(\d+)\s*m", expand=False),
        errors="coerce"
    )
    .fillna(0)
    .astype("Int64")
)

In [20]:
# remove unecessary columns that we already split or processed
movies_df = movies_df.drop(columns=["critics score","critics reviews","audience score","audience reviews","runtime","release date (theaters)"])

In [21]:
# simple renaming and organization
movies_df = movies_df.rename(columns={
    "Director": "director",
    "Producer": "producer",
    "Genre": "genre",
    "original language": "original_language",
 
})

In [22]:
len(movies_df)

3822

In [23]:
movies_df = movies_df.drop_duplicates(
    subset=["title", "release_date_theaters_year"],
    keep="first"
).reset_index(drop=True)

### No null values

In [24]:
movies_df.isna().sum()

title                          0
genre                          0
director                       0
original_language              0
summary                        0
critic_reviews_count           0
critic_score_percentage        0
audience_score_percentage      0
is_audience_rating_verified    0
audience_rating_count          0
release_date_theaters_year     0
release_date_theaters_month    0
release_date_theaters_day      0
runtime_hour                   0
runtime_minutes                0
dtype: int64

### No duplicates

In [25]:
duplicates = movies_df[
    movies_df.duplicated(
        subset=["title", "release_date_theaters_year"],
        keep=False
    )
]

duplicates

,title,genre,director,original_language,summary,critic_reviews_count,critic_score_percentage,audience_score_percentage,is_audience_rating_verified,audience_rating_count,release_date_theaters_year,release_date_theaters_month,release_date_theaters_day,runtime_hour,runtime_minutes


In [26]:
len(movies_df)

3311

In [27]:
movies_df.to_csv("../../data/processed/rotten_tomatoes_cleaned.csv", index=False)
